# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library. We demonstrate step-by-step operations including metadata exploration, record set listing, data loading, processing, and visualization.

### Dataset Source
The FAIR² dataset is provided via a Croissant schema URL:

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import matplotlib.pyplot as plt

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset via mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata
metadata = dataset.metadata.to_json()

print(f"{metadata['name']}:")
print(metadata['description'])

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

We use dataset metadata to inspect the available record sets, their fields, and column definitions. All references are by their `@id` fields for consistency.

In [ ]:
# List available record sets with @id and field information
record_sets = dataset.metadata.record_sets

print("Available record sets:")
for rs in record_sets:
    print(f"  RecordSet @id: {rs.id}")
    fields = rs.fields
    print("    Fields:")
    for field in fields:
        print(f"      Field @id: {field.id} | name: {field.name} | type: {field.data_type}")
        # Columns
        for col in field.columns:
            print(f"        Column @id: {col.id} | label: {col.label}")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

We'll extract a list of record set `@id`s dynamically, load each as a DataFrame, and print column IDs for inspection.

In [ ]:
# Extract record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for RecordSet @id: {record_set_id}")
        print("Columns:", df.columns.tolist())
        print(df.head())
        print()
    else:
        print(f"No records found for RecordSet @id: {record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we demonstrate filtering and normalization using a numeric field. Please adjust field `@id`s as needed based on your own overview results.

In [ ]:
# Choose a record set and numeric field @id for demonstration
if dataframes:
    selected_record_set_id = list(dataframes.keys())[0]
    df = dataframes[selected_record_set_id]
    
    # Attempt to select the first numeric column
    numeric_columns = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_columns:
        numeric_field_id = numeric_columns[0]
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]

        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt to group by first categorical column
        categorical_columns = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]
        group_field_id = categorical_columns[0] if categorical_columns else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
    else:
        print("No numeric columns available for EDA.")
else:
    print("No data frames available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of the selected numeric field and its grouped means if applicable.

In [ ]:
# Visualize histogram of the numeric field and bar plot of grouped means
if dataframes and 'numeric_field_id' in locals():
    plt.figure(figsize=(8,4))
    df[numeric_field_id].hist(bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if 'grouped_df' in locals():
        plt.figure(figsize=(8,4))
        plt.bar(grouped_df[group_field_id], grouped_df[numeric_field_id])
        plt.title(f"Mean of {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
In this notebook, we loaded and explored the FAIR² Croissant dataset using the `mlcroissant` library, processing data by referencing all entities with their `@id`s.

- We inspected the available record sets and fields.
- Loaded record sets into DataFrames for further analysis.
- Applied basic filtering, normalization, grouping, and visualizations.

This workflow provides a foundation for further statistical or modeling analyses using open, FAIR datasets annotated with Croissant.